# 02 Gradient Boosting Machines

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/blob/main/07-Machine-Learning/02_Gradient_Boosting_Machines.ipynb) [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/main?filepath=07-Machine-Learning/02_Gradient_Boosting_Machines.ipynb) [![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE) [![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)


In [ ]:
rng = np.random.default_rng(42)  # single reproducible generator

# === Environment Setup ===
import matplotlib.pyplot as plt
import numpy as np
import xgboost as xgb
from IPython.display import Image, Markdown, display
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# --- Configuration ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 14, 'figure.figsize': (10, 6), 'figure.dpi': 150})
%config InlineBackend.figure_format = 'retina'
np.set_printoptions(suppress=True, linewidth=120, precision=4)

# --- Utility Functions ---


---

### Table of Contents

1.  [**Boosting Intuition: Learning from Errors**](#intro)
2.  [**The Gradient Boosting Algorithm**](#algorithm)
3.  [**XGBoost: The Workhorse of Tabular Data**](#xgboost)
4.  [**Code Lab: Predicting House Prices with XGBoost**](#code-lab)
5.  [**Summary**](#summary)


## The Lens: Ensembles of Weak Learners
**What problem are we solving?**
A single decision tree is easy to interpret but prone to overfitting. How do we combine many weak models into one powerful predictor? **Gradient Boosting** builds an ensemble sequentially: each new tree corrects the errors of the previous ones by fitting the residual gradient.

**Why this method?**
XGBoost, LightGBM, and CatBoost dominate ML competitions and applied economics. They handle mixed data types, missing values, and non-linearities with minimal tuning—making them the go-to tool for prediction tasks in empirical economics.



**Economic question.** In *02 Gradient Boosting Machines*, what must remain economically invariant when the computational representation changes? For economists, predictive performance is useful but not sufficient. The model must be evaluated against the decision or forecasting problem, the information set available at prediction time, and the cost of distribution shift or leakage. Ask what inductive bias the method introduces, how tuning choices are validated out of sample, and which errors matter economically. When the goal is causal or structural, prediction should be treated as a nuisance component rather than evidence of identification by itself.

### Learning Objectives
* **Explain** the gradient boosting algorithm as sequential residual fitting.
* **Tune** key hyperparameters (learning rate, depth, number of rounds) using cross-validation.
* **Implement** XGBoost and LightGBM for regression and classification tasks.
* **Interpret** feature importance and partial dependence plots.

### Prerequisites
* **`07-Machine-Learning/01_Introduction_to_ML_for_Economists.ipynb`**: Bias-variance tradeoff, cross-validation, and decision trees.
* **`01-Foundations/12_NumPy.ipynb`**: Array manipulation.
* **`01-Foundations/13_Pandas.ipynb`**: Tabular data handling.
* **`01-Foundations/14_Matplotlib.ipynb`**: Plotting and diagnostics.


> **Learning path:** Building on [`01_Introduction_to_ML_for_Economists.ipynb`](01_Introduction_to_ML_for_Economists.ipynb); next continue with [`03_Support_Vector_Machines.ipynb`](03_Support_Vector_Machines.ipynb).


<a id='intro'></a>
## 1. Boosting Intuition: Learning from Errors

**Boosting** is an ensemble technique that builds models in a sequential fashion. Each new model is trained to correct the errors made by its predecessors. Unlike bagging, which focuses on reducing variance, boosting aims to reduce bias.

The core idea is to fit a sequence of weak learners (e.g., shallow decision trees) to weighted versions of the data, where more weight is given to the observations that were misclassified by earlier models.

![The Boosting Process](../images/07-Machine-Learning/boosting_process.png)


<a id='algorithm'></a>
## 2. The Gradient Boosting Algorithm

**Gradient Boosting** frames the boosting problem as a gradient descent optimization in function space. Each new weak learner is trained to fit the negative gradient of the loss function with respect to the predictions of the current ensemble. For squared error loss, this simplifies to fitting each new tree to the *residuals* (the errors) of the previous model.
> **Historical Context: Gradient Boosting**
> The Gradient Boosting algorithm was developed by Jerome Friedman in 1999. It is a generalization of the AdaBoost algorithm, and it allows for the use of arbitrary differentiable loss functions. This makes it a very flexible and powerful tool, and it is one of the most widely used machine learning algorithms today.


<a id='xgboost'></a>
## 3. XGBoost: The Workhorse of Tabular Data

**XGBoost (eXtreme Gradient Boosting)** is a highly efficient and effective implementation of the gradient boosting algorithm. It includes several key innovations:
- **Regularization:** It adds L1 and L2 regularization terms to the objective function to prevent overfitting.
- **Sparsity Awareness:** It can handle missing values efficiently.
- **Parallelization:** It can parallelize the construction of trees.

XGBoost is often the go-to algorithm for competitions and real-world applications involving tabular data.


<a id='code-lab'></a>
## 4. Code Lab: Predicting House Prices with XGBoost

Let's use XGBoost to predict house prices from a set of features.


### XGBoost for House Price Prediction


In [ ]:

# Generate synthetic data
X = rng.random(100, 5) * 10
y = 50 + (X[:, 0] * 1.5) + (X[:, 1] * 0.8) + (X[:, 2] * 2.1) + rng.standard_normal(100) * 5

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# We instantiate the XGBoost regressor, specifying the squared error loss function and the number of trees to build.
xg_reg = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, seed=42)
# We then fit the model to the training data.
xg_reg.fit(X_train, y_train)

# We can then use the trained model to make predictions on the test set.
y_pred = xg_reg.predict(X_test)
# We evaluate the model's performance using the root mean squared error.
rmse = np.sqrt(mean_squared_error(y_test, y_pred))


In [ ]:
display(Markdown(f"> **Note:** RMSE: {rmse:.4f}"))


In [ ]:

# XGBoost provides a built-in function to visualize the importance of each feature in the model.
xgb.plot_importance(xg_reg)
plt.title('Feature Importance')
plt.show()


## Exercises

**1. Mechanism and assumptions (Conceptual):** Explain the loss/objective and inductive bias of **02 Gradient Boosting Machines**. Distinguish optimization error, estimation error, and generalization error in the economic use case.

**2. Reproduce and diagnose (Applied):** Build a leakage-safe validation experiment using 1. Boosting Intuition: Learning from Errors, 2. The Gradient Boosting Algorithm. Compare a simple baseline with the featured method using an economically relevant metric and report uncertainty across folds or seeds.

**3. Robust extension (Challenge):** Stress-test the model under temporal, subgroup, or covariate distribution shift. Identify which performance degradation matters for the downstream economic decision and propose one mitigation without using the test set for tuning.

<details>
<summary>Solution guidance</summary>

A strong solution states assumptions before computation, includes an independent diagnostic or limiting-case check, and interprets the result in the units of the economic problem. For the challenge, separate changes caused by the economic assumption from changes caused by numerical approximation or tuning.

</details>


<a id='summary'></a>
## 5. Summary

Gradient Boosting Machines, and XGBoost in particular, are powerful and widely used models for tabular data. Their sequential, error-correcting nature makes them highly accurate, and implementations like XGBoost provide the efficiency and regularization needed for real-world applications.


In [ ]:
# --- Gradient Boosting with XGBoost ---
# pip install xgboost
import xgboost as xgb
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Generate Data (Credit Scoring Example)
n = 1000
X_credit = rng.normal(0, 1, (n, 10))
# Non-linear decision boundary
logits = 2 * X_credit[:, 0]**2 - 3 * np.sin(X_credit[:, 1]) + X_credit[:, 2] * X_credit[:, 3]
p = 1 / (1 + np.exp(-logits))
y_credit = np.random.binomial(1, p)

X_train, X_test, y_train, y_test = train_test_split(X_credit, y_credit, test_size=0.2)

# Train XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

params = {
    'max_depth': 3,
    'eta': 0.1,
    'objective': 'binary:logistic',
    'eval_metric': 'logloss'
}

bst = xgb.train(params, dtrain, num_boost_round=100)

# Evaluate
preds = bst.predict(dtest)
predictions = [round(value) for value in preds]
accuracy = accuracy_score(y_test, predictions)
print(f"XGBoost Accuracy: {accuracy * 100:.2f}%")

# Feature Importance
xgb.plot_importance(bst)
plt.show()


## References & Further Reading

- Hastie, T., Tibshirani, R. & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.). Springer.
- James, G., Witten, D., Hastie, T., Tibshirani, R. & Taylor, J. (2023). *An Introduction to Statistical Learning with Applications in Python*. Springer.
- Goodfellow, I., Bengio, Y. & Courville, A. (2016). *Deep Learning*. MIT Press.
